# Buenas prácticas

El notebook solo arranca Spark y llama a la clase. La configuración está en `parameter.json` y la lógica en `ETL.py`.

In [1]:
import json
import sys
from pathlib import Path

from pyspark.sql import SparkSession

CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "best_practices",
    Path.cwd() / "notebooks" / "best_practices",
    Path("/home/jovyan/notebooks/best_practices"),
]
ROOT = next(path for path in CANDIDATES if (path / "ETL.py").exists())
sys.path.insert(0, str(ROOT))

params = json.loads((ROOT / "parameter.json").read_text())
iceberg = params["iceberg"]

try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName(params["etl_name"])
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", iceberg["warehouse"])
    .getOrCreate()
)

In [2]:
from ETL import OrderItemsETL

target = OrderItemsETL(spark, params).run()
target

etl: order_items_resumen env=local
extract: public.orders
extract: public.order_items
load: local.db_lakehouse.order_items_staging


'local.db_lakehouse.order_items_staging'

In [3]:
spark.sql(f"SELECT * FROM {target}").show(5, truncate=False)

+--------------------------------+-------------+------------+------------------------+-----+-------------+-----+
|order_id                        |order_item_id|order_status|order_purchase_timestamp|price|freight_value|total|
+--------------------------------+-------------+------------+------------------------+-----+-------------+-----+
|f373335aac9a659de916f7170b8bc07a|1            |shipped     |2018-03-17 15:32:31     |35.9 |23.28        |59.18|
|118045506e1c1dda060171af43fe11b4|5            |delivered   |2018-03-08 19:06:05     |45.0 |27.08        |72.08|
|118045506e1c1dda060171af43fe11b4|4            |delivered   |2018-03-08 19:06:05     |45.0 |27.08        |72.08|
|118045506e1c1dda060171af43fe11b4|3            |delivered   |2018-03-08 19:06:05     |45.0 |27.08        |72.08|
|118045506e1c1dda060171af43fe11b4|2            |delivered   |2018-03-08 19:06:05     |45.0 |27.08        |72.08|
+--------------------------------+-------------+------------+------------------------+-----+----